In [4]:
import pandas as pd
from db_queries import get_location_metadata
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from tqdm import tqdm
import numpy as np
from matplotlib.ticker import MaxNLocator
from pathlib import Path
import yaml
import xarray as xr

# Set paths ####################################################################
RAW_DATA_PATH = "/mnt/team/rapidresponse/pub/population/modeling/climate_malnutrition/child_mortality/training_data/2026_10_07.01/data.parquet"
RESULTS_28q0_PATH = "/mnt/team/rapidresponse/pub/population/modeling/climate_malnutrition/neonatal_mortality/results/2026_10_08.01/ssp245_shifted.parquet"
RESULTS_5q28_PATH = "/mnt/team/rapidresponse/pub/population/modeling/climate_malnutrition/child_mortality/results/2026_10_16.01/ssp245_shifted.parquet"
RESULTS_5q0_SINGLE_RUN_PATH = "/mnt/team/rapidresponse/pub/population/modeling/climate_malnutrition/child_mortality/results/2026_10_01.01/ssp245_shifted.parquet"
GBD_5q28_FILE = "/mnt/team/rapidresponse/pub/population/modeling/climate_malnutrition/input/gbd_prevalence/gbd_mean_5q28_prevalence.parquet"
GBD_5q0_FILE = "/mnt/team/rapidresponse/pub/population/modeling/climate_malnutrition/input/gbd_prevalence/gbd_mean_child_mortality_prevalence.parquet"
GBD_28q0_FILE = "/mnt/team/rapidresponse/pub/population/modeling/climate_malnutrition/input/gbd_prevalence/gbd_mean_neonatal_mortality_prevalence.parquet"

PLOT_OUTPUT = "/mnt/team/rapidresponse/pub/population/modeling/climate_malnutrition/child_mortality/plots/"


df_raw = pd.read_parquet(RAW_DATA_PATH)

# Add location names
locs = get_location_metadata(location_set_id=39, release_id=16)

df_raw = df_raw.merge(
    locs[["ihme_loc_id", "location_name", "level", "super_region_name"]],
    on="ihme_loc_id",
    how="left",
)

df_raw.head()

,ihme_loc_id,geospatial_id,psu,strata,line_id,hh_id,lat,long,lbd_admin2_id,indv_id,...,age_36_m,age_48_m,age_60_m,age_group,lower_bound,upper_bound,consumption_median_RT,location_name,level,super_region_name
0,TJK,307,307,7,3,35,38.467862,68.826338,87095,341838_307_35_3_1_2009_4_2017_9_101_1_NA_NA,...,0,0,0,age_1_m,0,1,4.515261,Tajikistan,3,"Central Europe, Eastern Europe, and Central Asia"
1,NPL,343,343,13,1,42,28.654069,81.076586,93790,286782_343_42_1_2_2015_9_2016_8_11_1_19_NA,...,0,0,0,age_1_m,0,1,4.503675,Nepal,3,South Asia
2,IND,261103,261103,52,1,75,21.016793,83.319533,72207,157050_261103_75_1_1_2013_12_2015_5_29_1_NA_NA,...,0,0,0,age_1_m,0,1,4.236426,India,3,South Asia
3,ETH,285,285,12,1,112,10.500095,34.379398,71504,218568_285_112_1_1_2014_2_2016_2_24_1_NA_NA,...,0,0,0,age_1_m,0,1,3.509062,Ethiopia,3,Sub-Saharan Africa
4,PAK,507,507,9,1,23,35.515992,74.517198,71645,286783_507_23_1_2_2016_3_2017_2_23_1_NA_NA,...,0,0,0,age_1_m,0,1,4.312736,Pakistan,3,South Asia


In [5]:
# Recombine 28q0 and 5q28 into 5q0 #############################################
# The two series were modelled separately and shifted separately, and they
# partition childhood at day 28:
#
#   28q0  neonatal_mortality  P(die in days 0-27), from birth
#   5q28  child_mortality     P(die before age 5 | survived to day 28)
#
# They do NOT add. 28q0 is measured from birth and 5q28 is conditional on
# having survived the neonatal period, so the two are probabilities over
# different denominators. What composes is SURVIVAL:
#
#   l(5yr)/l(0) = [l(28d)/l(0)] x [l(5yr)/l(28d)]
#     1 - 5q0   =   (1 - 28q0)  x    (1 - 5q28)
#
# hence
#
#   5q0 = 1 - (1 - 28q0) * (1 - 5q28)
#
# Taking 28q0 + 5q28 instead would OVERSTATE under-5 mortality, because it
# charges the 5q28 share to the full birth cohort rather than to the neonatal
# survivors; the gap is the product term 28q0 * 5q28, which is largest exactly
# where mortality is highest.
#
# Applied draw by draw rather than to the means. That is sound here because the
# draw index is the CLIMATE draw index in both runs - inference passes `draw`
# straight to load_climate_raster - so draw i on both sides is the same climate
# realisation, and pairing them preserves that shared driver instead of
# inventing independence between the two models.
COMBINE_OUT_VERSION = "2026_10_20.01"
COMBINE_OUT_DIR = (
    Path("/mnt/team/rapidresponse/pub/population/modeling/climate_malnutrition")
    / "child_mortality"
    / "results"
    / COMBINE_OUT_VERSION
)
COMBINE_OUT_FILE = COMBINE_OUT_DIR / "ssp245_shifted.parquet"
# The completed direct 5q0 run, used only as a check - it is an independent
# estimate of the same quantity, so the recombination should land near it in
# the historical period where both are anchored to GBD.
COMBINE_CHECK_PATH = (
    "/mnt/team/rapidresponse/pub/population/modeling/climate_malnutrition/"
    "child_mortality/results/2026_10_01.01/ssp245_shifted.parquet"
)

# age_group_id differs between the inputs by construction (42 vs 1) and is the
# one key that must NOT be joined on; everything else has to match exactly.
_cmb_nn = pd.read_parquet(RESULTS_28q0_PATH).droplevel("age_group_id").sort_index()
_cmb_pn = pd.read_parquet(RESULTS_5q28_PATH).droplevel("age_group_id").sort_index()

if not _cmb_nn.index.equals(_cmb_pn.index):
    msg = (
        f"inputs do not align: {len(_cmb_nn):,} vs {len(_cmb_pn):,} rows, "
        f"{_cmb_nn.index.difference(_cmb_pn.index).size} keys only in 28q0"
    )
    raise ValueError(msg)
if list(_cmb_nn.columns) != list(_cmb_pn.columns):
    raise ValueError("draw columns differ between the two inputs")
for _nm, _df in (("28q0", _cmb_nn), ("5q28", _cmb_pn)):
    if _df.isna().any().any():
        raise ValueError(f"{_nm} contains nulls")

cmb_5q0 = 1.0 - (1.0 - _cmb_nn) * (1.0 - _cmb_pn)

# Invariants the identity guarantees; checked rather than assumed, because a
# silent sign or alignment error here would look like a plausible series.
assert ((cmb_5q0.to_numpy() >= 0) & (cmb_5q0.to_numpy() <= 1)).all(), "outside [0, 1]"
assert (cmb_5q0.to_numpy() >= _cmb_nn.to_numpy() - 1e-12).all(), "5q0 below 28q0"
assert (cmb_5q0.to_numpy() >= _cmb_pn.to_numpy() - 1e-12).all(), "5q0 below 5q28"
assert not cmb_5q0.isna().any().any(), "nulls in the result"

# Restore age_group_id as 1 (under 5), the id the rest of the pipeline uses for
# this quantity, and put the index back in the pipeline's order.
_cmb_idx = pd.MultiIndex.from_arrays(
    [
        cmb_5q0.index.get_level_values("location_id"),
        cmb_5q0.index.get_level_values("year_id"),
        cmb_5q0.index.get_level_values("sex_id"),
        np.full(len(cmb_5q0), 1, dtype="int64"),
        cmb_5q0.index.get_level_values("scenario"),
    ],
    names=["location_id", "year_id", "sex_id", "age_group_id", "scenario"],
)
cmb_5q0.index = _cmb_idx

COMBINE_OUT_DIR.mkdir(parents=True, exist_ok=True)
cmb_5q0.to_parquet(COMBINE_OUT_FILE)

# Provenance, because three different quantities now live in this results
# folder - direct 5q0 runs, 5q28 runs and recombinations - and nothing in the
# data tells them apart: all three carry age_group_id 1.
(COMBINE_OUT_DIR / "README.txt").write_text(
    f"""{COMBINE_OUT_VERSION} - under-5 mortality (5q0), RECOMBINED, not modelled directly.

5q0 = 1 - (1 - 28q0) * (1 - 5q28), applied draw by draw.

  28q0  {RESULTS_28q0_PATH}
  5q28  {RESULTS_5q28_PATH}

Both parents are post-residual (shifted) outputs. age_group_id is 1, the same
id the direct 5q0 runs and the 5q28 run use, so the version is the only thing
that identifies this as a recombination.

Written by notebooks/child_mortality/2026_10_20_child_mortality.ipynb.
"""
)

print(f"Saved {COMBINE_OUT_FILE}")
print(
    f"  {len(cmb_5q0):,} rows  "
    f"{cmb_5q0.index.get_level_values('location_id').nunique()} locations  "
    f"{len(cmb_5q0.columns)} draws  "
    f"range {cmb_5q0.to_numpy().min():.5f}-{cmb_5q0.to_numpy().max():.5f}"
)

# Check against the independent direct-5q0 run. Historical agreement is the
# real test: both are anchored to GBD there, so a recombination that has gone
# wrong shows up as a level shift. The forecast is allowed to differ - it comes
# from different models, and the direct run used an older SDI vintage.
_cmb_chk = pd.read_parquet(COMBINE_CHECK_PATH).droplevel("age_group_id").sort_index()
_cmb_j = cmb_5q0.droplevel("age_group_id").sort_index()
_shared = _cmb_j.index.intersection(_cmb_chk.index)
_r = _cmb_j.loc[_shared].mean(axis=1) / _cmb_chk.loc[_shared].mean(axis=1)
_r = _r.replace([np.inf, -np.inf], np.nan).dropna()
_yr = _r.index.get_level_values("year_id")
for _lab, _m in (
    ("historical (<=2023)", _yr <= 2023),
    ("forecast (>2023)", _yr > 2023),
):
    _s = _r[_m]
    print(
        f"  vs direct 5q0 2026_10_01.01, {_lab:19s} "
        f"p10 {_s.quantile(.1):.3f}  median {_s.median():.3f}  p90 {_s.quantile(.9):.3f}"
    )

Saved /mnt/team/rapidresponse/pub/population/modeling/climate_malnutrition/child_mortality/results/2026_10_20.01/ssp245_shifted.parquet
  95,546 rows  473 locations  100 draws  range 0.00000-0.29148
  vs direct 5q0 2026_10_01.01, historical (<=2023) p10 0.953  median 0.998  p90 1.015
  vs direct 5q0 2026_10_01.01, forecast (>2023)    p10 0.890  median 0.981  p90 1.171


In [7]:
# GBD, FHS and the modeled mortality windows, pre- and post-shift ############
# One panel per location. Seventeen series in five groups, each group carrying
# its own visual language so the groups never have to compete for a hue:
#
#   GBD              solid, achromatic (black -> dark grey -> grey)
#   FHS              solid / dashed, red; external forecasts, 2024-2100 only
#   Modeled shifted  dashed, one hue per window
#   Modeled residual dash-dot, same hue as its shifted counterpart
#   Modeled original dotted, same hue as its shifted counterpart, thinner
#
# The three modeled groups are the three stages each run passes through, and
# sev_means.parquet stores all of them: `model_value` is the raw inference
# output (original), `residual_model_value` is that plus the fitted
# GBD-vs-model residual, and the shifted draws on disk are the final stage.
# Reading them as dotted -> dash-dot -> dashed shows what each step moved.
#
# The two 5q0 series share the blue hue because they are the same quantity by
# two routes - the single-run model, and the combination of the other two. To
# keep them apart without spending a fourth categorical hue, the constructed
# line carries a long dash, double the weight, and a ring marker every decade;
# the single-run line is a fine short dash.
#
# GBD and FHS already publish every level of the hierarchy, so they are read
# straight off. The modeled series exist only at the 473 most-detailed
# locations and are population-weighted up `path_to_top_parent`, all four using
# ONE weight - the under-5 population - so the constructed identity survives
# aggregation (max absolute drift 1.7e-03, median relative 1e-09). At
# most-detailed level it is exact by construction.
#
# Location labels carry a crossing flag: the first year after 2023 at which the
# modeled 5q0 single-run falls BELOW the modeled 28q0. That ordering is
# impossible - neonatal deaths are a subset of under-5 deaths - so it marks the
# locations where the problem this work set out to fix is still present.
import html as _cmp_html
import json as _cmp_json

CMP_PATH = f"{PLOT_OUTPUT}gbd_vs_modeled_mortality_windows.html"
CMP_POP_FUT = "/mnt/share/forecasting/data/32/future/population/future_population_s130v41/population.nc"
CMP_POP_PAST = "/mnt/share/forecasting/data/16/past/population/20250603_etl_run_id_417/population.nc"
CMP_POP_SCENARIO = 130
CMP_U5_PARTS = [2, 3, 388, 389, 238, 34]
CMP_ANCHOR = 2023
CMP_SEXES = [1, 2]

# key, label, colour, dash, width, group, marker-every-N-years (0 = none), paint order
CMP_SERIES = [
    ("gbd_5q0", "GBD 5q0", "#0b0b0b", "none", 2.4, "GBD", 0, 11),
    ("gbd_5q28", "GBD 5q28", "#52514e", "none", 2.0, "GBD", 0, 10),
    ("gbd_28q0", "GBD 28q0", "#898781", "none", 2.0, "GBD", 0, 9),
    ("fhs_with_cuts", "FHS 5q0, with ODA cuts", "#d1495b", "none", 2.0, "FHS", 0, 7),
    ("fhs_no_cuts", "FHS 5q0, without ODA cuts", "#d1495b", "7 4", 2.0, "FHS", 0, 6),
    ("pre_28q0", "Original 28q0", "#eb6834", "1 3", 1.5, "Modeled, original", 0, 1),
    ("pre_5q28", "Original 5q28", "#1baf7a", "1 3", 1.5, "Modeled, original", 0, 2),
    (
        "pre_5q0_single",
        "Original 5q0 single-run",
        "#2a78d6",
        "1 3",
        1.5,
        "Modeled, original",
        0,
        3,
    ),
    (
        "pre_5q0_constructed",
        "Original 5q0 constructed",
        "#2a78d6",
        "1 3",
        2.4,
        "Modeled, original",
        20,
        4,
    ),
    # The residual stage sits between the two: model_value plus the fitted
    # GBD-vs-model residual, before the scenario shift. Same hue per window as
    # its neighbours, dash-dot so the three stages read as dotted -> dash-dot
    # -> dashed. Paint order 8 puts them above FHS and below GBD.
    ("res_28q0", "Residual 28q0", "#eb6834", "7 2 1 2", 1.7, "Modeled, residual", 0, 5),
    ("res_5q28", "Residual 5q28", "#1baf7a", "7 2 1 2", 1.7, "Modeled, residual", 0, 5),
    (
        "res_5q0_single",
        "Residual 5q0 single-run",
        "#2a78d6",
        "7 2 1 2",
        1.7,
        "Modeled, residual",
        0,
        5,
    ),
    (
        "res_5q0_constructed",
        "Residual 5q0 constructed",
        "#2a78d6",
        "7 2 1 2",
        2.6,
        "Modeled, residual",
        15,
        5,
    ),
    ("mod_28q0", "Modeled 28q0", "#eb6834", "6 3", 2.0, "Modeled, shifted", 0, 12),
    ("mod_5q28", "Modeled 5q28", "#1baf7a", "6 3", 2.0, "Modeled, shifted", 0, 13),
    (
        "mod_5q0_single",
        "Modeled 5q0 single-run",
        "#2a78d6",
        "3 3",
        1.7,
        "Modeled, shifted",
        0,
        14,
    ),
    (
        "mod_5q0_constructed",
        "Modeled 5q0 constructed",
        "#2a78d6",
        "14 5",
        3.4,
        "Modeled, shifted",
        10,
        15,
    ),
]

# shifted draws -> mean, keyed by (shifted draws, pre-shift column in
# sev_means, residual-stage column in sev_means)
CMP_MOD_SOURCES = {
    "mod_28q0": (RESULTS_28q0_PATH, "pre_28q0", "res_28q0"),
    "mod_5q28": (RESULTS_5q28_PATH, "pre_5q28", "res_5q28"),
    "mod_5q0_single": (RESULTS_5q0_SINGLE_RUN_PATH, "pre_5q0_single", "res_5q0_single"),
    "mod_5q0_constructed": (str(COMBINE_OUT_FILE), None, None),
}
CMP_GBD_PATHS = {
    "gbd_5q0": GBD_5q0_FILE,
    "gbd_28q0": GBD_28q0_FILE,
    "gbd_5q28": GBD_5q28_FILE,
}
CMP_FHS = {
    "fhs_with_cuts": "/mnt/share/forecasting/data/32/future/death/20260717_q0_reference_s130v108_noreagg_nq0",
    "fhs_no_cuts": "/mnt/share/forecasting/data/32/future/death/20260716_q0_reference_s130v98_nq0",
}

_cmp_root = "/mnt/team/rapidresponse/pub/population/modeling/climate_malnutrition/"
_cmp_hier = pd.read_parquet(_cmp_root + "input/fhs_hierarchy.parquet")
_cmp_keys = ["location_id", "year_id", "sex_id"]

# ---- modeled: shifted (draw means), original and residual (sev_means) -------
_cmp_mod = None
for _k, (_p, _pre, _res) in CMP_MOD_SOURCES.items():
    _d = pd.read_parquet(_p).mean(axis=1).rename(_k).reset_index()
    _d = _d.drop(columns=[c for c in ("age_group_id", "scenario") if c in _d.columns])
    _cmp_mod = _d if _cmp_mod is None else _cmp_mod.merge(_d, on=_cmp_keys, how="outer")
    if _pre or _res:
        _sv = pd.read_parquet(Path(_p).parent / "sev_means.parquet").reset_index()
        _want = {"model_value": _pre, "residual_model_value": _res}
        _want = {src: dst for src, dst in _want.items() if dst}
        _sv = _sv[_cmp_keys + list(_want)].rename(columns=_want)
        _cmp_mod = _cmp_mod.merge(_sv, on=_cmp_keys, how="left")
# the constructed series has no saved pre-shift, so build it the same way its
# shifted counterpart was built - from the two parents' originals
_cmp_mod["pre_5q0_constructed"] = 1.0 - (1.0 - _cmp_mod["pre_28q0"]) * (
    1.0 - _cmp_mod["pre_5q28"]
)
# same for the residual stage, from the two parents' residual-stage values
_cmp_mod["res_5q0_constructed"] = 1.0 - (1.0 - _cmp_mod["res_28q0"]) * (
    1.0 - _cmp_mod["res_5q28"]
)

_cmp_modcols = [c for c in _cmp_mod.columns if c not in _cmp_keys]
if _cmp_mod[_cmp_modcols].isna().any().any():
    raise ValueError(
        f"modeled inputs have gaps: {_cmp_mod[_cmp_modcols].isna().sum().to_dict()}"
    )

# ---- population-weight the modeled series up the hierarchy -----------------
_cmp_pop_parts = []
for _p, _sel in ((CMP_POP_PAST, {}), (CMP_POP_FUT, {"scenario": CMP_POP_SCENARIO})):
    _d = (
        xr.open_dataset(_p)
        .sel(age_group_id=CMP_U5_PARTS, sex_id=CMP_SEXES, **_sel)
        .to_dataframe()
        .reset_index()
    )
    _v = "population" if "population" in _d.columns else "draws"
    _cmp_pop_parts.append(
        _d.groupby(_cmp_keys, as_index=False)[_v]
        .sum()
        .rename(columns={_v: "population"})
    )
_cmp_pop = pd.concat(_cmp_pop_parts).drop_duplicates(_cmp_keys, keep="last")

_cmp_det = _cmp_hier[_cmp_hier.most_detailed == 1][
    ["location_id", "path_to_top_parent"]
]
_cmp_f = _cmp_mod.merge(_cmp_det, on="location_id").merge(_cmp_pop, on=_cmp_keys)
_cmp_e = _cmp_f.assign(agg_id=_cmp_f.path_to_top_parent.str.split(",")).explode(
    "agg_id"
)
_cmp_e["agg_id"] = _cmp_e.agg_id.astype(int)
for _c in _cmp_modcols:
    _cmp_e[_c] = _cmp_e[_c] * _cmp_e["population"]
_cmp_agg = _cmp_e.groupby(["agg_id", "year_id", "sex_id"], as_index=False)[
    _cmp_modcols + ["population"]
].sum(min_count=1)
for _c in _cmp_modcols:
    _cmp_agg[_c] = _cmp_agg[_c] / _cmp_agg["population"]
_cmp_agg = _cmp_agg.rename(columns={"agg_id": "location_id"}).drop(columns="population")

# ---- GBD and FHS: already at every level of the hierarchy ------------------
_cmp_gbd = None
for _k, _p in CMP_GBD_PATHS.items():
    _d = pd.read_parquet(_p).reset_index().drop(columns="age_group_id")
    _d = _d.rename(columns={"gbd_mean_prevalence": _k})
    _cmp_gbd = _d if _cmp_gbd is None else _cmp_gbd.merge(_d, on=_cmp_keys, how="outer")

# year5q0 is the under-5 analogue of day28q0, stored at age_group_id 1 on the
# same probability scale as gbd_value, so it needs no rescale.
_cmp_fhs = None
for _k, _p in CMP_FHS.items():
    _d = (
        xr.open_dataarray(f"{_p}/year5q0.nc")
        .sel(scenario=CMP_POP_SCENARIO, sex_id=CMP_SEXES)
        .mean(dim="draw")
        .squeeze()
        .to_dataframe()
        .reset_index()
        .rename(columns={"value": _k})
    )
    _d = _d[_cmp_keys + [_k]]
    _cmp_fhs = _d if _cmp_fhs is None else _cmp_fhs.merge(_d, on=_cmp_keys, how="outer")
print(
    f"FHS overlay: {_cmp_fhs.location_id.nunique()} locations, "
    f"{int(_cmp_fhs.year_id.min())}-{int(_cmp_fhs.year_id.max())}"
)

_cmp_all = _cmp_agg.merge(_cmp_gbd, on=_cmp_keys, how="left").merge(
    _cmp_fhs, on=_cmp_keys, how="left"
)
_cmp_years = sorted(_cmp_all.year_id.unique())
_cmp_nm = _cmp_hier.set_index("location_id")["location_name"]
_cmp_lv = _cmp_hier.set_index("location_id")["level"]
_cmp_sr = _cmp_hier.set_index("location_id")["super_region_name"]
_CMP_LV_NAME = {
    0: "Global",
    1: "Super region",
    2: "Region",
    3: "Country",
    4: "Subnational",
}

_cmp_locs, _cmp_ncross = {}, 0
for _lid, _g in _cmp_all.groupby("location_id"):
    if _lid not in _cmp_lv.index:
        continue
    _by_sex, _cross = {}, {}
    for _sex, _gs in _g.groupby("sex_id"):
        _gs = _gs.set_index("year_id")
        _ser = {}
        for _k, *_rest in CMP_SERIES:
            _col = _gs[_k].reindex(_cmp_years) if _k in _gs.columns else None
            _ser[_k] = (
                [None] * len(_cmp_years)
                if _col is None
                else [None if pd.isna(v) else round(float(v), 7) for v in _col]
            )
        _by_sex[str(int(_sex))] = _ser
        # first year after the anchor where under-5 drops below neonatal
        _bad = _gs.index[
            (_gs.index > CMP_ANCHOR) & (_gs["mod_5q0_single"] < _gs["mod_28q0"])
        ]
        _cross[str(int(_sex))] = int(_bad.min()) if len(_bad) else None
    if any(v is not None for v in _cross.values()):
        _cmp_ncross += 1
    _cmp_locs[str(_lid)] = {
        "n": str(_cmp_nm.get(_lid)),
        "lv": int(_cmp_lv.get(_lid)),
        "lvn": _CMP_LV_NAME.get(int(_cmp_lv.get(_lid)), "?"),
        "sr": (None if pd.isna(_cmp_sr.get(_lid)) else str(_cmp_sr.get(_lid))),
        "x": _cross,
        "bySex": _by_sex,
    }

CMP_PAYLOAD = {
    "years": [int(y) for y in _cmp_years],
    "anchor": CMP_ANCHOR,
    "series": [
        {"k": k, "l": l, "c": c, "d": d, "w": w, "g": g, "m": m}
        for k, l, c, d, w, g, m, _o in sorted(CMP_SERIES, key=lambda s: s[7])
    ],
    "locs": _cmp_locs,
}
print(
    f"comparison payload: {len(CMP_PAYLOAD['locs'])} locations, "
    f"{len(CMP_PAYLOAD['years'])} years, {len(CMP_PAYLOAD['series'])} series; "
    f"{_cmp_ncross} locations where the single-run 5q0 still crosses below 28q0 after {CMP_ANCHOR}"
)

CMP_HTML = r"""<!doctype html>
<html lang="en"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Mortality windows: GBD, FHS and the models</title>
<style>
:root{color-scheme:light;
  --plane:#f9f9f7; --surface:#fcfcfb; --ink:#0b0b0b; --ink2:#52514e; --muted:#898781;
  --grid:#e1e0d9; --axis:#c3c2b7; --ring:rgba(11,11,11,.10); --accent:#2a78d6;}
@media (prefers-color-scheme:dark){:root:where(:not([data-theme="light"])){
  color-scheme:dark;
  --plane:#0d0d0d; --surface:#1a1a19; --ink:#fff; --ink2:#c3c2b7; --muted:#898781;
  --grid:#2c2c2a; --axis:#383835; --ring:rgba(255,255,255,.10); --accent:#3987e5;}}
:root[data-theme="dark"]{color-scheme:dark;
  --plane:#0d0d0d; --surface:#1a1a19; --ink:#fff; --ink2:#c3c2b7; --muted:#898781;
  --grid:#2c2c2a; --axis:#383835; --ring:rgba(255,255,255,.10); --accent:#3987e5;}
*{box-sizing:border-box}
body{margin:0;background:var(--plane);color:var(--ink);
  font:14px/1.5 system-ui,-apple-system,"Segoe UI",sans-serif}
header{padding:14px 16px;border-bottom:1px solid var(--ring);display:flex;
  align-items:baseline;gap:12px;flex-wrap:wrap}
h1{font-size:17px;margin:0;font-weight:650}
.sub{color:var(--ink2);font-size:12.5px}.spacer{flex:1}
button,select,input{font:inherit;color:var(--ink);background:var(--surface);
  border:1px solid var(--ring);border-radius:7px;padding:6px 10px}
button{cursor:pointer}button:hover{border-color:var(--muted)}
main{display:grid;grid-template-columns:270px 1fr;gap:16px;padding:16px;align-items:start}
@media(max-width:860px){main{grid-template-columns:1fr}}
.panel{background:var(--surface);border:1px solid var(--ring);border-radius:11px;padding:12px}
#list{max-height:70vh;overflow:auto;margin-top:8px}
.row{padding:6px 8px;border-radius:6px;cursor:pointer;display:flex;gap:8px;
  align-items:baseline;font-size:13px}
.row:hover{background:var(--plane)}
.row[aria-selected="true"]{background:var(--accent);color:#fff}
.row .lv{color:var(--muted);font-size:11px}
.row[aria-selected="true"] .lv{color:rgba(255,255,255,.75)}
#chart{width:100%;height:clamp(360px,54vh,560px)}
.lg{display:inline-flex;align-items:center;gap:6px;margin:0 10px 6px 0;font-size:12.5px;
  background:none;border:none;padding:2px 4px;border-radius:5px;cursor:pointer;color:var(--ink)}
.lg[aria-pressed="false"]{opacity:.35}
.lgs{display:flex;flex-wrap:wrap;margin-top:8px;align-items:center}
.lgh{font-size:11px;color:var(--muted);text-transform:uppercase;letter-spacing:.06em;
  margin:0 8px 6px 0;align-self:center;background:none;border:none;cursor:pointer;padding:2px 4px}
.lgh:hover{color:var(--ink)}
.row .xf{color:#d1495b;font-size:11px}
.row[aria-selected="true"] .xf{color:#ffd9df}
#tip{position:fixed;pointer-events:none;opacity:0;transition:opacity .08s;
  background:var(--surface);border:1px solid var(--ring);border-radius:9px;
  padding:8px 10px;font-size:12.5px;box-shadow:0 6px 24px rgba(0,0,0,.16);z-index:9}
#tip table{border-collapse:collapse;margin-top:4px}
#tip td{padding:1px 4px;white-space:nowrap}
#tip td.v{text-align:right;font-variant-numeric:tabular-nums}
footer{padding:0 16px 20px;color:var(--ink2);font-size:12.5px;max-width:980px}
</style></head><body>
<header>
  <h1>Mortality windows: GBD, FHS and the models</h1>
  <div class="sub">ssp245 &middot; draw means &middot; 5q0 constructed = 1 &minus; (1 &minus; 28q0)(1 &minus; 5q28)</div>
  <div class="spacer"></div>
  <label class="sub" for="sex">Sex</label>
  <select id="sex"><option value="1">Male</option><option value="2">Female</option></select>
  <button id="theme">Theme</button>
</header>
<main>
  <div class="panel">
    <input id="q" type="search" placeholder="Search locations" style="width:100%">
    <select id="lvl" style="width:100%;margin-top:8px">
      <option value="">All levels</option><option value="0">Global</option>
      <option value="1">Super region</option><option value="2">Region</option>
      <option value="3">Country</option><option value="4">Subnational</option>
    </select>
    <div id="list"></div>
  </div>
  <div class="panel">
    <div id="title" style="font-weight:650"></div>
    <div class="sub" id="subtitle"></div>
    <svg id="chart"></svg>
    <div class="lgs" id="legend"></div>
  </div>
</main>
<div id="tip"></div>
<footer>
  <div>Solid achromatic lines are GBD; dashed coloured lines are modeled. The two
  5q0 series share a hue because they are the same quantity by two routes &mdash;
  the single-run model, and the constructed combination of the other two modeled
  series. GBD stops at 2024; the vertical mark is the anchor year, where the
  shifted models meet GBD by construction.</div>
  <div style="margin-top:6px">Modeled series are population-weighted up the
  hierarchy using one common weight, the under-5 population, so the constructed
  identity survives aggregation. Click a legend entry to hide a series, or a
  group heading to hide the group.</div>
  <div style="margin-top:6px"><b style="color:#d1495b">(crosses YYYY)</b> beside a
  location is the first year after the anchor at which the modeled 5q0 single-run
  falls below the modeled 28q0. Neonatal deaths are a subset of under-5 deaths, so
  that ordering cannot happen; the flag marks where it still does. It is computed
  for the selected sex.</div>
</footer>
<script>
const D = __PAYLOAD__;
const S = D.series, YEARS = D.years;
const css = n => {
  const v = getComputedStyle(document.documentElement).getPropertyValue(n).trim();
  if (!v) throw new Error('undefined CSS variable: ' + n);
  return v;
};
const el = (t, a) => { const e = document.createElementNS('http://www.w3.org/2000/svg', t);
  for (const k in a) e.setAttribute(k, a[k]); return e; };
const esc = s => String(s).replace(/[&<>"]/g, c => ({'&':'&amp;','<':'&lt;','>':'&gt;','"':'&quot;'}[c]));
const fmt = v => v == null ? '&mdash;' : (v * 1000).toFixed(2) + ' /1000';

const ids = Object.keys(D.locs);
let sex = '1', cur = ids.includes('1') ? '1' : ids[0], hidden = new Set(), geom = null;

const filtered = () => {
  const q = document.getElementById('q').value.trim().toLowerCase();
  const lv = document.getElementById('lvl').value;
  return ids.filter(i => {
    const L = D.locs[i];
    if (lv !== '' && String(L.lv) !== lv) return false;
    return !q || L.n.toLowerCase().includes(q);
  }).sort((a, b) => D.locs[a].lv - D.locs[b].lv || D.locs[a].n.localeCompare(D.locs[b].n));
};

function renderList(){
  const box = document.getElementById('list'); box.innerHTML = '';
  const f = filtered();
  if (f.length && !f.includes(cur)) { cur = f[0]; }
  f.forEach(i => {
    const L = D.locs[i];
    const r = document.createElement('div');
    r.className = 'row'; r.setAttribute('role', 'option');
    r.setAttribute('aria-selected', i === cur ? 'true' : 'false');
    const xy = L.x ? L.x[sex] : null;
    r.innerHTML = `<span>${esc(L.n)}${xy ? ` <span class="xf">(crosses ${xy})</span>` : ''}</span>`
      + `<span style="flex:1"></span><span class="lv">${esc(L.lvn)}</span>`;
    r.onclick = () => { cur = i; renderList(); draw(); };
    box.appendChild(r);
  });
  if (!f.length) box.innerHTML = '<div class="sub" style="padding:8px">No matches</div>';
}

function visible(){ return S.filter(s => !hidden.has(s.k)); }

function draw(){
  const L = D.locs[cur];
  document.getElementById('title').textContent = L.n;
  document.getElementById('subtitle').textContent =
    L.lvn + (L.sr ? ' · ' + L.sr : '') + ' · ' + (sex === '1' ? 'Male' : 'Female');
  const data = (L.bySex && L.bySex[sex]) || {};
  const svg = document.getElementById('chart');
  svg.innerHTML = '';
  const W = svg.clientWidth || 820, H = svg.clientHeight || 440;
  const M = {t: 12, r: 16, b: 30, l: 62};
  const vals = [];
  visible().forEach(s => (data[s.k] || []).forEach(v => { if (v != null) vals.push(v); }));
  if (!vals.length) { geom = null; return; }
  const ymax = Math.max(...vals) * 1.06, ymin = 0;
  const x = y => M.l + (y - YEARS[0]) / (YEARS[YEARS.length-1] - YEARS[0]) * (W - M.l - M.r);
  const Y = v => v == null ? null : H - M.b - (v - ymin) / (ymax - ymin) * (H - M.t - M.b);
  geom = {x, Y, W, H, M, data};

  const gridc = css('--grid'), axisc = css('--axis'), ink2 = css('--ink2');
  const ticks = 5;
  for (let i = 0; i <= ticks; i++){
    const v = ymin + (ymax - ymin) * i / ticks, yy = Y(v);
    svg.appendChild(el('line', {x1: M.l, x2: W - M.r, y1: yy, y2: yy, stroke: gridc, 'stroke-width': 1}));
    const t = el('text', {x: M.l - 8, y: yy + 4, 'text-anchor': 'end', fill: ink2, 'font-size': 11});
    t.textContent = (v * 1000).toFixed(1); svg.appendChild(t);
  }
  const ylab = el('text', {x: 14, y: H / 2, fill: ink2, 'font-size': 11,
    transform: `rotate(-90 14 ${H/2})`, 'text-anchor': 'middle'});
  ylab.textContent = 'Deaths per 1000'; svg.appendChild(ylab);
  [YEARS[0], 2025, 2050, 2075, YEARS[YEARS.length-1]].forEach(yr => {
    if (yr < YEARS[0] || yr > YEARS[YEARS.length-1]) return;
    const t = el('text', {x: x(yr), y: H - 10, 'text-anchor': 'middle', fill: ink2, 'font-size': 11});
    t.textContent = yr; svg.appendChild(t);
  });
  svg.appendChild(el('line', {x1: M.l, x2: W - M.r, y1: H - M.b, y2: H - M.b, stroke: axisc, 'stroke-width': 1}));
  svg.appendChild(el('line', {x1: x(D.anchor), x2: x(D.anchor), y1: M.t, y2: H - M.b,
    stroke: css('--muted'), 'stroke-width': 1.1, 'stroke-dasharray': '4 4'}));

  visible().forEach(s => {
    const arr = data[s.k] || []; let d = '', pen = false;
    YEARS.forEach((yv, i) => {
      const v = arr[i], yy = Y(v);
      if (v == null || yy == null) { pen = false; return; }
      d += (pen ? 'L' : 'M') + x(yv).toFixed(1) + ' ' + yy.toFixed(1) + ' '; pen = true;
    });
    if (d) svg.appendChild(el('path', {d, fill: 'none', stroke: s.c, 'stroke-width': s.w,
      'stroke-dasharray': s.d, 'stroke-linecap': 'round', 'stroke-linejoin': 'round'}));
    // ring markers: the only thing separating the two 5q0 series at a glance
    if (s.m) YEARS.forEach((yv, i) => {
      if (yv % s.m) return;
      const v = arr[i], yy = Y(v); if (v == null || yy == null) return;
      svg.appendChild(el('circle', {cx: x(yv), cy: yy, r: 3.6, fill: css('--surface'),
        stroke: s.c, 'stroke-width': 1.8}));
    });
  });
}

function renderLegend(){
  const box = document.getElementById('legend'); box.innerHTML = '';
  let group = null;
  S.slice().sort((a, b) => a.g.localeCompare(b.g)).forEach(s => {
    if (s.g !== group){ group = s.g;
      const h = document.createElement('button'); h.className = 'lgh'; h.textContent = group;
      h.title = 'Show or hide this whole group';
      const keys = S.filter(z => z.g === h.textContent).map(z => z.k);
      h.onclick = () => { const anyOn = keys.some(k => !hidden.has(k));
        keys.forEach(k => anyOn ? hidden.add(k) : hidden.delete(k));
        renderLegend(); draw(); };
      box.appendChild(h); }
    const b = document.createElement('button');
    b.className = 'lg'; b.setAttribute('aria-pressed', hidden.has(s.k) ? 'false' : 'true');
    b.innerHTML = `<svg width="26" height="10"><line x1="1" y1="5" x2="25" y2="5"
      stroke="${s.c}" stroke-width="${s.w}" stroke-dasharray="${s.d}" stroke-linecap="round"/>${
      s.m ? `<circle cx="13" cy="5" r="3.2" fill="${css('--surface')}" stroke="${s.c}" stroke-width="1.6"/>` : ''
      }</svg>${esc(s.l)}`;
    b.onclick = () => { hidden.has(s.k) ? hidden.delete(s.k) : hidden.add(s.k);
      renderLegend(); draw(); };
    box.appendChild(b);
  });
}

const tip = document.getElementById('tip');
document.getElementById('chart').addEventListener('mousemove', ev => {
  if (!geom) return;
  const r = ev.currentTarget.getBoundingClientRect();
  const px = ev.clientX - r.left;
  let best = YEARS[0], bd = Infinity;
  YEARS.forEach(yv => { const d = Math.abs(geom.x(yv) - px); if (d < bd){ bd = d; best = yv; } });
  const i = YEARS.indexOf(best);
  let rows = '';
  visible().forEach(s => {
    const v = (geom.data[s.k] || [])[i];
    rows += `<tr><td><svg width="22" height="8"><line x1="1" y1="4" x2="21" y2="4"
      stroke="${s.c}" stroke-width="${s.w}" stroke-dasharray="${s.d}" stroke-linecap="round"/></svg>
      ${esc(s.l)}</td><td class="v">${fmt(v)}</td></tr>`;
  });
  tip.innerHTML = `<b>${esc(D.locs[cur].n)} &middot; ${best}</b><table>${rows}</table>`;
  tip.style.opacity = 1;
  const tw = tip.offsetWidth, th = tip.offsetHeight;
  tip.style.left = Math.min(ev.clientX + 14, innerWidth - tw - 8) + 'px';
  tip.style.top = Math.max(8, Math.min(ev.clientY - th / 2, innerHeight - th - 8)) + 'px';
});
document.getElementById('chart').addEventListener('mouseleave', () => { tip.style.opacity = 0; });

document.getElementById('q').addEventListener('input', () => { renderList(); draw(); });
document.getElementById('lvl').addEventListener('change', () => { renderList(); draw(); });
document.getElementById('sex').addEventListener('change', e => {
  sex = e.target.value; renderList(); draw(); });
document.getElementById('theme').onclick = () => {
  const now = document.documentElement.getAttribute('data-theme');
  const dark = now ? now === 'dark' : matchMedia('(prefers-color-scheme: dark)').matches;
  document.documentElement.setAttribute('data-theme', dark ? 'light' : 'dark');
  renderLegend(); draw();
};
addEventListener('resize', () => { if (geom) draw(); });
renderList(); renderLegend(); draw();
</script></body></html>
"""

Path(CMP_PATH).write_text(
    CMP_HTML.replace(
        "__PAYLOAD__", _cmp_json.dumps(CMP_PAYLOAD, separators=(",", ":"))
    ),
    encoding="utf-8",
)
print(f"Saved {CMP_PATH} ({Path(CMP_PATH).stat().st_size/1e6:.2f} MB)")

FHS overlay: 538 locations, 2024-2050
comparison payload: 513 locations, 101 years, 13 series; 216 locations where the single-run 5q0 still crosses below 28q0 after 2023
Saved /mnt/team/rapidresponse/pub/population/modeling/climate_malnutrition/child_mortality/plots/gbd_vs_modeled_mortality_windows.html (11.45 MB)
